# Lab W6: EDA dan Leakage Check pada PathMNIST

## Pre-flight Checklist

> [!IMPORTANT]
> Konsep yang ditandai (§) merujuk ke `06_W6_Representations_Temporal_Leakage.md`.

**Yang Anda butuhkan sebelum mulai:**
- Bab W6 sudah dibaca, terutama §2.1 (EDA 3 lapis), §2.2 (5 jenis leakage), §2.3 (audit label), §2.4 (pemahaman preprocessing leakage).
- `pip install -e '.[medical]'` untuk MedMNIST.
- Familiar dengan pandas, matplotlib, hashlib.

**Yang akan Anda hasilkan di akhir lab:**
- EDA 3 lapis: shape/integritas → distribusi kelas → relasi warna-kelas.
- Audit train-test overlap dengan MD5 hash. **Audit label kualitas mandatory** (lihat §2.3 W6).
- Verifikasi preprocessing fit-only-on-train.
- Audit report 1 halaman sebagai paragraf, bukan centang.

**Resource:**
- **Hardware:** CPU cukup. Total runtime ~5-10 menit (download dataset + EDA).
- **Storage:** ~200 MB (PathMNIST).
- **Estimasi waktu kerja:** 3-4 jam termasuk inspeksi visual dan menulis audit report.

**Pendamping:** Bab W6 di `06_W6_Representations_Temporal_Leakage.md`.

## Tujuan
1. Download PathMNIST (histologi 9 kelas) dan lakukan EDA 3 lapis.
2. Audit 5 jenis data leakage.
3. Verifikasi bahwa preprocessing fit hanya pada train set.
4. Tulis audit report yang jujur, bukan centang formalitas.

## Prasyarat
```bash
pip install -e '.[medical]'
```

## Daftar Periksa
- [ ] EDA 3 lapis: shape/statistik, distribusi kelas, relasi warna-kelas.
- [ ] Audit train-test overlap (hash-based).
- [ ] Verifikasi mean/std dihitung dari train only.
- [ ] **Audit label kualitas (mandatory)**: inspeksi 50 sampel acak ATAU 20 confident-wrong setelah baseline training (lihat §2.3 W6).
- [ ] Audit report ditulis sebagai paragraf, bukan centang.

## 0. Setup dan muat data

In [ ]:
# ── Setup ────────────────────────────────────────────────────────────────────
# Di Google Colab: clone repo, cd ke template_repo/, lalu jalankan sel ini.
import sys, os
_root = os.path.abspath('..')
if _root not in sys.path:
    sys.path.insert(0, _root)

# medmnist tidak tersedia by default di Google Colab
import importlib
if importlib.util.find_spec('medmnist') is None:
    import subprocess
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'medmnist'])

print("Root:", _root)

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt
from src.data import build_loaders

cfg = {
    'name': 'pathmnist',
    'root': '../data',
    'image_size': 28,
    'batch_size': 256,
    'num_workers': 0,
    'val_split': 0.0,  # pakai split resmi PathMNIST
    'augment': False,
}
train_loader, val_loader, test_loader = build_loaders(cfg)
print(f'train: {len(train_loader.dataset)} | val: {len(val_loader.dataset)} | test: {len(test_loader.dataset)}')

## 1. EDA Lapis 1: Shape dan Statistik Dasar

In [ ]:
# Ambil 1 batch dan periksa properti dasar
x_batch, y_batch = next(iter(train_loader))
y_batch = torch.as_tensor(y_batch).long().view(-1)

print('=== Shape dan Tipe ===')
print(f'Input shape (batch): {x_batch.shape}')
print(f'Label shape (batch): {y_batch.shape}')
print(f'Dtype: {x_batch.dtype}')
print(f'Label dtype: {y_batch.dtype}')

print('\n=== Statistik Pixel (batch pertama) ===')
print(f'Min:  {x_batch.min():.4f}')
print(f'Max:  {x_batch.max():.4f}')
print(f'Mean (per channel): {x_batch.mean(dim=[0,2,3]).numpy().round(4)}')
print(f'Std  (per channel): {x_batch.std(dim=[0,2,3]).numpy().round(4)}')

print('\n=== Label unik ===')
print(f'Label values: {torch.unique(y_batch).numpy()}')

In [ ]:
# Hitung statistik pixel dari seluruh train set (penting untuk normalisasi)
# Ini yang harus dipakai sebagai mean/std normalisasi  -  BUKAN dari val/test

channel_sum = torch.zeros(3)
channel_sq_sum = torch.zeros(3)
n_pixels = 0

for x, _ in train_loader:
    b, c, h, w = x.shape
    channel_sum += x.mean(dim=[0, 2, 3]) * b
    channel_sq_sum += (x ** 2).mean(dim=[0, 2, 3]) * b
    n_pixels += b

mean_train = channel_sum / n_pixels
std_train = torch.sqrt(channel_sq_sum / n_pixels - mean_train ** 2)

print('=== Mean/Std dari TRAIN set (untuk normalisasi) ===')
print(f'mean: {mean_train.numpy().round(4)}')
print(f'std:  {std_train.numpy().round(4)}')
print('\n⚠ Pastikan nilai ini yang dipakai di src/data.py untuk PathMNIST,')
print('  bukan hardcode dari dataset lain!')

## 2. EDA Lapis 2: Distribusi Kelas

In [ ]:
# Hitung distribusi kelas di train, val, test
from collections import Counter

def count_classes(loader, n_classes=9):
    counts = Counter()
    for _, y in loader:
        y = torch.as_tensor(y).long().view(-1)
        counts.update(y.numpy().tolist())
    return [counts.get(i, 0) for i in range(n_classes)]

train_counts = count_classes(train_loader)
val_counts   = count_classes(val_loader)
test_counts  = count_classes(test_loader)

PATHMNIST_CLASSES = [
    'adipose', 'background', 'debris', 'lymphocytes',
    'mucus', 'smooth muscle', 'normal colon', 'cancer stroma', 'colorectal adenocarcinoma'
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, counts, title in zip(axes, [train_counts, val_counts, test_counts], ['Train', 'Val', 'Test']):
    bars = ax.bar(range(9), counts, color='steelblue', alpha=0.8)
    ax.set_xticks(range(9))
    ax.set_xticklabels([c[:8] for c in PATHMNIST_CLASSES], rotation=45, ha='right', fontsize=8)
    ax.set_title(f'{title} (n={sum(counts)})')
    ax.set_ylabel('Jumlah sampel')
    # Annotate jumlah
    for bar, count in zip(bars, counts):
        ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 50,
                str(count), ha='center', va='bottom', fontsize=7)

plt.suptitle('Distribusi Kelas PathMNIST', fontsize=11)
plt.tight_layout()
plt.show()

# Laporan imbalance
total_train = sum(train_counts)
print('\nProposi kelas di train set:')
for i, (cls, cnt) in enumerate(zip(PATHMNIST_CLASSES, train_counts)):
    pct = cnt / total_train * 100
    flag = '⚠' if pct < 5 else ' '
    print(f'  {flag} {i}: {cls:30s} {cnt:6d} ({pct:5.1f}%)')

## 3. EDA Lapis 3: Relasi Warna dan Kelas

In [ ]:
# Visualisasi sampel per kelas
class_samples = {i: [] for i in range(9)}
for x, y in train_loader:
    y = torch.as_tensor(y).long().view(-1)
    for img, label in zip(x, y):
        label = label.item()
        if len(class_samples[label]) < 3:
            class_samples[label].append(img)
    if all(len(v) >= 3 for v in class_samples.values()):
        break

# Denormalisasi
mean_t = torch.tensor(mean_train).view(3, 1, 1)
std_t  = torch.tensor(std_train).view(3, 1, 1)

fig, axes = plt.subplots(9, 3, figsize=(6, 18))
for row, (cls_id, imgs) in enumerate(class_samples.items()):
    for col, img in enumerate(imgs[:3]):
        denorm = (img * std_t + mean_t).clamp(0, 1).permute(1, 2, 0).numpy()
        axes[row][col].imshow(denorm)
        axes[row][col].axis('off')
        if col == 0:
            axes[row][col].set_ylabel(PATHMNIST_CLASSES[cls_id][:12], rotation=0,
                                       labelpad=70, fontsize=8, va='center')
plt.suptitle('3 Contoh per Kelas PathMNIST', fontsize=11)
plt.tight_layout()
plt.savefig('../experiments/pathmnist_samples.png', dpi=100, bbox_inches='tight')
plt.show()

In [ ]:
# Apakah kelas bisa dipisahkan hanya dari intensitas rata-rata?
# Hitung mean intensitas per channel per kelas

class_intensities = {i: [] for i in range(9)}
for x, y in train_loader:
    y = torch.as_tensor(y).long().view(-1)
    for img, label in zip(x, y):
        mean_rgb = img.mean(dim=[1, 2]).numpy()  # shape (3,)
        class_intensities[label.item()].append(mean_rgb)

fig, axes = plt.subplots(1, 3, figsize=(14, 5))
channel_names = ['Red', 'Green', 'Blue']
for ch_idx, (ax, ch_name) in enumerate(zip(axes, channel_names)):
    data_per_class = [np.array([v[ch_idx] for v in class_intensities[i]]) for i in range(9)]
    ax.boxplot(data_per_class, labels=[f'{i}' for i in range(9)])
    ax.set_xlabel('Kelas')
    ax.set_ylabel('Mean intensity')
    ax.set_title(f'Channel {ch_name}')
    ax.grid(alpha=0.3)

plt.suptitle('Distribusi Intensitas per Kelas PathMNIST', fontsize=11)
plt.tight_layout()
plt.show()

print('Pertanyaan: Kelas mana yang paling berbeda secara warna? Apakah perbedaan itu cukup untuk klasifikasi sederhana?')

## 4. Audit Leakage: Train-Test Overlap

In [ ]:
import hashlib

def tensor_hash(t: torch.Tensor) -> str:
    """Hash tensor bytes  -  cepat untuk mendeteksi duplikasi exact."""
    return hashlib.md5(t.numpy().tobytes()).hexdigest()

print('Menghitung hash train split...')
train_hashes = set()
for x, _ in train_loader:
    for img in x:
        train_hashes.add(tensor_hash(img))

print('Menghitung hash test split...')
overlap_count = 0
total_test = 0
for x, _ in test_loader:
    for img in x:
        total_test += 1
        if tensor_hash(img) in train_hashes:
            overlap_count += 1

print(f'\nTotal test samples: {total_test}')
print(f'Train-test overlap (exact): {overlap_count}')
if overlap_count == 0:
    print('✓ Tidak ada duplikasi exact antara train dan test.')
else:
    print(f'⚠ Ada {overlap_count} sampel test yang identik dengan train! Ini data leakage.')

## 4b. Audit Label: Inconsistency Detection

Langkah wajib dari §2.3 W6: deteksi near-duplicate images dengan label berbeda. Dua gambar yang hampir identik tapi punya label beda = indikasi pelabelan yang tidak konsisten. Pakai image hashing (MD5 + Hamming distance pada perceptual hash) untuk mendeteksi pasangan suspect.

In [ ]:
# Deteksi near-duplicate dengan average hash (ahash) — hash berbasis konten visual
from PIL import Image
import hashlib
from collections import defaultdict

def ahash(img, hash_size=8):
    """Average hash: resize ke hash_size*hash_size, grayscale, threshold ke mean."""
    img = img.resize((hash_size, hash_size), Image.LANCZOS).convert('L')
    pixels = np.array(img).flatten()
    mean = pixels.mean()
    return ''.join('1' if p > mean else '0' for p in pixels)

def hamming_distance(h1, h2):
    return sum(c1 != c2 for c1, c2 in zip(h1, h2))

# Kumpulkan semua sampel dari train set dengan ahash
print('Menghitung ahash untuk semua sampel train...')
hashes = []
labels_list = []
all_images = []  # simpan numpy array untuk inspeksi visual nanti
for x, y in train_loader:
    y = torch.as_tensor(y).long().view(-1)
    for i in range(len(x)):
        img_pil = T.ToPILImage()(x[i])
        hashes.append(ahash(img_pil))
        labels_list.append(y[i].item())
        all_images.append(x[i].numpy())
    if len(hashes) >= len(train_loader.dataset):
        break

hashes = np.array(hashes)
labels_list = np.array(labels_list)
print(f'Total sampel: {len(hashes)}')

# Deteksi near-duplicate: Hamming distance ≤ 4 (threshold longgar untuk ahash)
print('\nMencari near-duplicate pairs dengan label berbeda...')
suspect_pairs = []
for i in range(min(len(hashes), 2000)):  # batasi ke 2000 sampel pertama untuk kecepatan
    for j in range(i + 1, min(len(hashes), 2000)):
        if hamming_distance(hashes[i], hashes[j]) <= 4:
            if labels_list[i] != labels_list[j]:
                suspect_pairs.append((i, j, labels_list[i], labels_list[j]))
            if len(suspect_pairs) >= 50:  # cukup untuk inspeksi
                break
    if len(suspect_pairs) >= 50:
        break

print(f'Ditemukan {len(suspect_pairs)} near-duplicate dengan label berbeda')
if len(suspect_pairs) > 0:
    print('\n=== 10 Pasangan Suspect Pertama ===')
    for idx, (i, j, li, lj) in enumerate(suspect_pairs[:10]):
        dist = hamming_distance(hashes[i], hashes[j])
        print(f'  Pasangan {idx+1}: idx ({i},{j}) label ({li},{lj}) Hamming={dist}')
else:
    print('✓ Tidak ditemukan near-duplicate dengan label berbeda dalam 2000 sampel.')
    print('  Ini menunjukkan konsistensi pelabelan yang baik — sebutkan di audit report.')

In [ ]:
# Inspeksi visual 10 pasangan suspect (jika ditemukan)
CLASSES = ['adipose','background','debris','lymphocytes','mucosa',
           'smooth muscle','normal colon mucosa','cancer-associated stroma','colorectal adenocarcinoma epithelium']

if len(suspect_pairs) > 0:
    n_show = min(5, len(suspect_pairs))
    fig, axes = plt.subplots(n_show, 2, figsize=(6, n_show*2.5))
    if n_show == 1:
        axes = axes[np.newaxis, :]
    for row, (i, j, li, lj) in enumerate(suspect_pairs[:n_show]):
        for col, (idx, lbl) in enumerate([(i, li), (j, lj)]):
            ax = axes[row, col]
            img = all_images[idx].transpose(1, 2, 0)  # CHW -> HWC
            img = np.clip(img, 0, 1) if img.max() <= 1 else img / 255.0
            ax.imshow(img, cmap='gray' if img.shape[-1] == 1 else None)
            ax.set_title(f'Label: {CLASSES[lbl] if lbl < len(CLASSES) else lbl}', fontsize=9)
            ax.axis('off')
    plt.suptitle('Near-Duplicate dengan Label Berbeda — Periksa Manual', fontsize=11)
    plt.tight_layout()
    plt.show()
else:
    print('Tidak ada suspect pairs untuk divisualisasi.')
    print('\nBerdasarkan audit label ini: PathMNIST tampaknya punya konsistensi pelabelan yang baik.')
    print('Sebutkan di audit report bahwa near-duplicate detection dengan ahash tidak menemukan inkonsistensi.')

### Analisis Label Inconsistency

Setelah inspeksi visual pasangan suspect:

- **Apakah ada pasangan yang jelas-jelas kelas berbeda tapi gambar hampir identik?** Jika ya, ini adalah label noise yang perlu ditangani (report ke pengelola dataset atau bersihkan).
- **Apakah near-duplicate adalah gambar yang sama difoto ulang?** Di data medis, ini mungkin slice histologi yang berdekatan — wajar punya label berbeda jika slice menunjukkan transisi jaringan.
- **Jika tidak ditemukan suspect:** sebutkan di audit report bahwa dataset lolos audit label inconsistency dengan metode ahash. Ini adalah hasil positif yang perlu didokumentasikan.

## 5. Audit Leakage: Preprocessing Pipeline

In [ ]:
# Verifikasi bahwa normalisasi yang dipakai di data.py menggunakan
# statistik dari train set, bukan dari seluruh dataset atau hardcode dari CIFAR-10

import inspect
from src import data as data_module

source = inspect.getsource(data_module)

# Cari apakah ada referensi ke PathMNIST-specific normalization
print('=== Cek src/data.py untuk normalisasi PathMNIST ===')
lines_with_normalize = [l.strip() for l in source.split('\n') if 'Normalize' in l or 'pathmnist' in l.lower()]
for line in lines_with_normalize:
    print(' ', line)

print()
print('Statistik dari train set yang dihitung di atas:')
print(f'  mean: {mean_train.numpy().round(4)}')
print(f'  std:  {std_train.numpy().round(4)}')
print()
print('Catatan: Jika src/data.py menggunakan nilai CIFAR-10 untuk PathMNIST,')
print('itu adalah preprocessing leakage (menggunakan asumsi dataset berbeda).')

## 5b. Baseline Training: +- Augmentation (5 Epoch)

Latih SimpleCNN ringan pada PathMNIST untuk membandingkan train/val gap dengan dan tanpa augmentasi. Ini bukan training penuh - hanya 5 epoch untuk mendeteksi apakah augmentasi bisa mengurangi overfitting.

In [ ]:
from src.models import SimpleCNN
import torch.nn as nn

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def train_baseline(train_loader, val_loader, epochs=5, label='baseline'):
    model = SimpleCNN(num_classes=9).to(device)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
    crit = nn.CrossEntropyLoss()
    hist = {'train_loss': [], 'train_acc': [], 'val_loss': [], 'val_acc': []}
    for epoch in range(epochs):
        model.train()
        total_loss, correct, total = 0.0, 0, 0
        for x, y in train_loader:
            y = torch.as_tensor(y).long().view(-1)
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            logits = model(x)
            loss = crit(logits, y)
            loss.backward()
            opt.step()
            total_loss += loss.item() * len(x)
            correct += (logits.argmax(dim=1) == y).sum().item()
            total += len(x)
        hist['train_loss'].append(total_loss / total)
        hist['train_acc'].append(correct / total)
        model.eval()
        val_loss, val_correct, val_total = 0.0, 0, 0
        with torch.no_grad():
            for x, y in val_loader:
                y = torch.as_tensor(y).long().view(-1)
                x, y = x.to(device), y.to(device)
                logits = model(x)
                val_loss += crit(logits, y).item() * len(x)
                val_correct += (logits.argmax(dim=1) == y).sum().item()
                val_total += len(x)
        hist['val_loss'].append(val_loss / val_total)
        hist['val_acc'].append(val_correct / val_total)
        gap = hist['train_acc'][-1] - hist['val_acc'][-1]
        print(f'{label:>20s} E{epoch+1} | tr_loss={hist["train_loss"][-1]:.4f} tr_acc={hist["train_acc"][-1]:.4f} va_acc={hist["val_acc"][-1]:.4f} gap={gap:+.4f}')
    return model, hist

# Baseline 1: Tanpa augmentasi
print('=== Baseline 1: Tanpa Augmentasi ===')
model_noaug, hist_noaug = train_baseline(train_loader, val_loader, epochs=5, label='NoAug')

# Baseline 2: Dengan augmentasi
cfg_aug = {**cfg, 'augment': True}
train_loader_aug, val_loader_aug, test_loader_aug = build_loaders(cfg_aug)
print('\n=== Baseline 2: Dengan Augmentasi ===')
model_aug, hist_aug = train_baseline(train_loader_aug, val_loader_aug, epochs=5, label='Augmented')

In [ ]:
# Plot perbandingan train/val gap
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].plot(hist_noaug['train_loss'], 'o-', color='steelblue', alpha=0.7, label='train (no aug)')
axes[0].plot(hist_noaug['val_loss'], 'o-', color='tomato', alpha=0.7, label='val (no aug)')
axes[0].plot(hist_aug['train_loss'], 's--', color='steelblue', label='train (+aug)')
axes[0].plot(hist_aug['val_loss'], 's--', color='tomato', label='val (+aug)')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].set_title('Loss Comparison')
axes[0].legend(); axes[0].grid(alpha=0.3)

axes[1].plot(np.array(hist_noaug['train_acc'])*100, 'o-', color='steelblue', alpha=0.7, label='train (no aug)')
axes[1].plot(np.array(hist_noaug['val_acc'])*100, 'o-', color='tomato', alpha=0.7, label='val (no aug)')
axes[1].plot(np.array(hist_aug['train_acc'])*100, 's--', color='steelblue', label='train (+aug)')
axes[1].plot(np.array(hist_aug['val_acc'])*100, 's--', color='tomato', label='val (+aug)')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy (%)'); axes[1].set_title('Accuracy Comparison')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()

# Tabel ringkasan
print(f'{"Condition":>20s} {"Train Acc":>10s} {"Val Acc":>10s} {"Gap":>10s}')
print('-' * 52)
noaug_gap = hist_noaug['train_acc'][-1] - hist_noaug['val_acc'][-1]
aug_gap = hist_aug['train_acc'][-1] - hist_aug['val_acc'][-1]
print(f'{"No Augmentation":>20s} {hist_noaug["train_acc"][-1]*100:9.1f}% {hist_noaug["val_acc"][-1]*100:9.1f}% {noaug_gap*100:+9.1f}%')
print(f'{"With Augmentation":>20s} {hist_aug["train_acc"][-1]*100:9.1f}% {hist_aug["val_acc"][-1]*100:9.1f}% {aug_gap*100:+9.1f}%')

print()
if aug_gap < noaug_gap:
    print(f'Augmentasi mengurangi gap dari {noaug_gap*100:.1f}% ke {aug_gap*100:.1f}% - efektif mengurangi overfitting')
else:
    print(f'Augmentasi TIDAK mengurangi gap (dari {noaug_gap*100:.1f}% ke {aug_gap*100:.1f}%). Coba augmentasi lebih kuat.')
    print('Catat temuan ini di audit report.')

## 6. Audit Leakage: Temporal dan Group (Patient)

Dua jenis leakage ini relevan untuk data medis:

### 6a. Temporal Leakage

PathMNIST tidak menyertakan timestamp gambar dalam metadata publik. Oleh karena itu:

- Kita tidak dapat memverifikasi apakah gambar di test set diambil setelah train set.
- Ini adalah **keterbatasan yang harus disebutkan di laporan**, bukan disembunyikan.
- Implikasi: jika model diterapkan ke data real, distribusi histologi bisa berubah seiring waktu (concept drift).

### 6b. Group Leakage (Patient-Level)

Idealnya, semua patch dari satu pasien harus masuk ke split yang sama (train ATAU test, tidak keduanya). Jika satu pasien punya patch di train dan test, model bisa belajar pasien-spesifik, bukan histologi-generik.

In [ ]:
# Cek apakah MedMNIST menyediakan patient metadata
try:
    import medmnist
    from medmnist import PathMNIST
    ds = PathMNIST(split='train', download=False, root='../data')
    # Cek atribut yang tersedia
    attrs = [a for a in dir(ds) if not a.startswith('_')]
    print('Atribut PathMNIST dataset:', attrs)
    if hasattr(ds, 'info'):
        print('\nInfo:', ds.info)
except Exception as e:
    print(f'Tidak bisa mengakses medmnist langsung: {e}')

print()
print('Catatan: MedMNIST v2 tidak menyertakan patient ID dalam dataset publik.')
print('Ini adalah keterbatasan yang perlu disebutkan di audit report.')

## 7. Audit Report

Tulis sebagai paragraf jujur di bawah. Bukan centang formalitas  -  tulis apa yang kamu temukan dan apa artinya.

### Audit Report PathMNIST

> *[Contoh format  -  ganti dengan temuanmu sendiri]*
>
> **Ringkasan dataset:** PathMNIST berisi X sampel train, Y sampel test, 9 kelas histologi kolorektal. Distribusi kelas tidak seimbang: kelas terbesar (adipose) menempati ~X% sampel, kelas terkecil (debris) hanya ~X%.
>
> **Audit overlap:** Tidak ditemukan duplikasi exact (hash MD5) antara train dan test split. Kemungkinan near-duplicate tidak diperiksa dalam lab ini karena membutuhkan embedding similarity  -  ini keterbatasan audit.
>
> **Audit preprocessing:** Normalisasi menggunakan mean/std yang dihitung dari train set saja (diverifikasi dari src/data.py). Tidak ada preprocessing leakage yang terdeteksi.
>
> **Audit temporal:** PathMNIST tidak menyediakan timestamp  -  temporal leakage tidak dapat diverifikasi. Ini keterbatasan yang perlu disebutkan di laporan eksperimen.
>
> **Audit group (patient):** Patient ID tidak tersedia di dataset publik. Split dilakukan di level patch, bukan patient. Artinya model bisa belajar fitur per-pasien, dan generalisasi ke pasien baru mungkin lebih rendah dari yang dilaporkan metrik ini.
>
> **Kesimpulan untuk eksperimen:** Dataset aman dipakai dengan catatan: (1) metrik harus diinterpretasi sebagai patch-level accuracy, bukan patient-level; (2) imbalance kelas perlu ditangani dengan stratified sampling atau weighted loss.

## 8. Refleksi

1. Dari tiga EDA lapis yang dilakukan, lapisan mana yang paling mengejutkan? Apa satu fakta tentang data yang tidak kamu duga sebelum melihatnya?

2. Audit patient-level leakage tidak bisa dilakukan karena data tidak tersedia. Bayangkan kamu adalah reviewer paper yang melihat hasil akurasi tinggi pada PathMNIST tanpa audit ini  -  apa satu pertanyaan yang akan kamu ajukan?

3. Jika kamu mengetahui ada 5% label yang mungkin salah di train set, apa yang akan kamu lakukan: (a) abaikan karena 5% kecil, (b) bersihkan semua, atau (c) eksperimen dengan dan tanpa pembersihan? Jelaskan alasanmu.

### Jawaban Refleksi

**1. Temuan paling mengejutkan:**
> *[tulis di sini]*

**2. Pertanyaan untuk paper tanpa patient audit:**
> *[tulis di sini]*

**3. Strategi menghadapi 5% label noise:**
> *[tulis di sini]*